# Part 2 — MST: Kruskal's & Prim's Algorithm
**Algorithm Analysis and Simulation Toolkit | Term 2, SY 2025–2026**

---
## Algorithm Explanations

**Kruskal's Algorithm**
Greedy approach that processes edges in order of weight. Uses Union-Find to detect cycles.
- Time Complexity: O(E log E) — dominated by sorting edges
- Space Complexity: O(V) — for Union-Find structure

**Prim's Algorithm**
Greedy approach that grows the MST from a starting vertex. Uses a min-heap to always pick the cheapest edge.
- Time Complexity: O(E log V) — with binary heap
- Space Complexity: O(V + E)

---
## Algorithm Flowcharts

### Kruskal's Algorithm
```
START
  └─► Sort all edges by weight (ascending)
  └─► Initialize Union-Find for V vertices
  └─► MST = []
  └─► For each edge (u, v, w) in sorted order:
        └─► find(u) == find(v)?
              │YES → SKIP (would form a cycle)
              │NO  → union(u, v); add edge to MST
        └─► len(MST) == V-1? → DONE
  └─► Output MST edges and total cost
```

### Prim's Algorithm
```
START
  └─► Initialize visited = {start_vertex}
  └─► Push all edges from start into min-heap
  └─► MST = []
  └─► While heap not empty:
        └─► Pop minimum weight edge (w, u, v)
        └─► v already visited? → SKIP
        └─► Mark v as visited
        └─► Add edge (u, v, w) to MST
        └─► Push all unvisited neighbors of v into heap
  └─► Output MST edges and total cost
```

### Union-Find (used by Kruskal's)
```
find(x): path compression
  └─► parent[x] == x? → return x
  └─► else: parent[x] = find(parent[x]); return parent[x]

union(x, y):
  └─► rx = find(x);  ry = find(y)
  └─► rx == ry? → return False  (same component = cycle)
  └─► rank-based merge to keep tree flat
  └─► return True
```

---
## Program Instructions
1. Run **Cell 2** to load all MST algorithms.
2. Enter vertices and edges manually, or click **Random Graph** to generate one.
3. Set the starting vertex for Prim's algorithm.
4. Toggle **Show step-by-step trace** to see detailed simulation.
5. Click **Run Both Algorithms** to execute and display results.
6. Run **Test Cases** to verify both algorithms produce the same MST cost.


In [1]:
import heapq, random
import ipywidgets as widgets
from collections import defaultdict
from IPython.display import display, clear_output

# ── UNION-FIND (Disjoint Set Union) ──────────────────────────────────────────
# Supports Kruskal's cycle detection. Uses path compression + union by rank
# for near-constant time operations.
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank   = [0] * n

    def find(self, x):
        # path compression: point directly to root on the way back up
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry: return False   # same component — adding this edge = cycle
        # union by rank keeps the tree shallow
        if self.rank[rx] < self.rank[ry]: rx, ry = ry, rx
        self.parent[ry] = rx
        if self.rank[rx] == self.rank[ry]: self.rank[rx] += 1
        return True

# ── INPUT VALIDATION ──────────────────────────────────────────────────────────
def validate_graph(vertices, edges):
    vset = set(vertices)
    if len(vertices) < 2: raise ValueError('Need at least 2 vertices.')
    for u, v, w in edges:
        if u not in vset or v not in vset:
            raise ValueError(f'Edge ({u},{v}) references vertex not in list.')
        if u == v: raise ValueError(f'Self-loop on vertex {u} is not allowed.')
        if w < 0:  raise ValueError(f'Negative weight {w} on edge ({u}--{v}).')

# ── GRAPH STRUCTURE DISPLAY ───────────────────────────────────────────────────
def display_graph_structure(vertices, edges):
    # builds adjacency list and prints it — spec requirement
    adj = defaultdict(list)
    for u, v, w in edges:
        adj[u].append((v, w)); adj[v].append((u, w))
    print('Graph Structure (Adjacency List):')
    print(f'  Vertices : {vertices}')
    print(f'  Edges    : {len(edges)}')
    print()
    for v in vertices:
        neighbors = ', '.join(f'{nb}(w={wt})' for nb, wt in sorted(adj[v]))
        print(f'  {v:>3} → {neighbors if neighbors else "(no edges)"}')
    print()
    print('Edge List (sorted by weight):')
    for u, v, w in sorted(edges, key=lambda e: e[2]):
        print(f'  {u} -- {v}  (weight {w})')
    print()

# ── KRUSKAL'S ALGORITHM ───────────────────────────────────────────────────────
# Greedy: picks cheapest edge that doesn't form a cycle, until MST is complete.
def kruskal(vertices, edges, verbose=True):
    validate_graph(vertices, edges)
    n = len(vertices)
    v_idx = {v: i for i, v in enumerate(vertices)}
    uf = UnionFind(n)
    mst = []
    sorted_edges = sorted(edges, key=lambda e: e[2])

    if verbose:
        print('=== KRUSKAL\'S ALGORITHM ===')
        print()
        display_graph_structure(vertices, edges)
        print('Edge Sorting (ascending by weight):')
        for u, v, w in sorted_edges:
            print(f'  ({u}--{v}, w={w})')
        print()
        print('Edge Selection + Cycle Detection:')

    step = 0
    for u, v, w in sorted_edges:
        added = uf.union(v_idx[u], v_idx[v])
        if added:
            mst.append((u, v, w)); step += 1
            if verbose:
                print(f'  Step {step}: ({u}--{v}, w={w})  →  ADDED     '
                      f'| MST so far: {[(a,b,c) for a,b,c in mst]}')
        else:
            if verbose:
                print(f'         ({u}--{v}, w={w})  →  SKIPPED   '
                      f'| Would form a cycle (find({u})==find({v}))')
        if len(mst) == n - 1: break

    cost = sum(w for _, _, w in mst)
    if verbose:
        print()
        print('MST Formation complete.')
        print()
        print('Edges selected for MST:')
        for u, v, w in mst:
            print(f'  {u} -- {v}  (weight {w})')
        print(f'Total MST Cost = {cost}')
    return mst, cost

# ── PRIM'S ALGORITHM ──────────────────────────────────────────────────────────
# Grows the MST from a start vertex, always expanding via the cheapest border edge.
def prim(vertices, edges, start=None, verbose=True):
    validate_graph(vertices, edges)
    adj = defaultdict(list)
    for u, v, w in edges: adj[u].append((v, w)); adj[v].append((u, w))

    if start is None: start = vertices[0]
    if start not in set(vertices): raise ValueError(f'Start vertex {start!r} not found.')

    visited = set(); mst = []
    heap = [(0, None, start)]

    if verbose:
        print('=== PRIM\'S ALGORITHM ===')
        print()
        display_graph_structure(vertices, edges)
        print(f'Starting vertex: {start}')
        print()
        print('Growing MST step by step:')

    step = 0
    while heap:
        w, u, v = heapq.heappop(heap)
        if v in visited: continue
        visited.add(v)
        if u is not None:
            mst.append((u, v, w)); step += 1
            if verbose:
                print(f'  Step {step}: Add ({u}--{v}, w={w})'
                      f'  | Visited: {sorted(visited)}')
        for nb, wt in adj[v]:
            if nb not in visited: heapq.heappush(heap, (wt, v, nb))

    cost = sum(c for _, _, c in mst)
    if verbose:
        print()
        print('Edges selected for MST:')
        for u, v, w in mst:
            print(f'  {u} -- {v}  (weight {w})')
        print(f'Total MST Cost = {cost}')
    return mst, cost

# ── RANDOM GRAPH GENERATOR ────────────────────────────────────────────────────
# Creates a random connected undirected weighted graph.
# Guarantees connectivity via a spanning chain before adding extra edges.
def random_graph(n, extra=None):
    if n < 2: raise ValueError('Need at least 2 vertices.')
    if extra is None: extra = n
    vertices = list(range(1, n + 1))
    eset = {}
    for i in range(1, n):  # spanning chain
        eset[(i, i+1)] = random.randint(1, 20)
    attempts = 0
    while len(eset) < n - 1 + extra and attempts < 300:
        attempts += 1
        u, v = random.randint(1, n), random.randint(1, n)
        if u != v:
            k = (min(u,v), max(u,v))
            if k not in eset: eset[k] = random.randint(1, 20)
    return vertices, [(u, v, w) for (u, v), w in eset.items()]

print('=== Part 2: MST Simulation ===')
print('  Kruskal\'s and Prim\'s algorithms loaded.')
print('  Scroll down to run the simulation and test cases.')


=== Part 2: MST Simulation ===
  Kruskal's and Prim's algorithms loaded.
  Scroll down to run the simulation and test cases.


## MST Simulation
Enter a graph manually or generate a random one, then run both algorithms.

In [2]:
# ── MST SIMULATION WIDGET ─────────────────────────────────────────────────────
# Enter vertices and edges manually or generate a random graph.
# Shows: graph structure, step-by-step simulation, final MST output.

w_verts = widgets.Text(
    value='1, 2, 3, 4, 5, 6',
    description='Vertices (csv):', layout=widgets.Layout(width='340px'),
    style={'description_width': 'initial'}
)
w_edges = widgets.Textarea(
    value='1 2 4\n1 3 3\n2 3 5\n2 4 6\n3 4 7\n3 5 8\n4 5 9\n4 6 5\n5 6 6',
    description='Edges (u v w):',
    layout=widgets.Layout(width='340px', height='175px'),
    style={'description_width': 'initial'}
)
w_start = widgets.Text(
    value='1', description="Prim's start:",
    layout=widgets.Layout(width='160px'), style={'description_width': 'initial'}
)
w_steps = widgets.Checkbox(value=True, description='Show step-by-step trace')
w_run   = widgets.Button(
    description='▶  Run Both Algorithms',
    button_style='success', layout=widgets.Layout(width='210px', height='36px')
)
w_rand_n = widgets.IntSlider(
    value=6, min=4, max=12, description='Random vertices:',
    style={'description_width': 'initial'}
)
w_rand = widgets.Button(
    description='🎲  Random Graph',
    button_style='warning', layout=widgets.Layout(width='150px', height='36px')
)
out_mst = widgets.Output()

def parse_graph():
    vertices = [int(x.strip()) for x in w_verts.value.split(',') if x.strip()]
    edges = []
    for line in w_edges.value.strip().splitlines():
        p = line.strip().split()
        if not p: continue
        if len(p) != 3: raise ValueError(f'Edge line must be "u v weight", got: {line!r}')
        edges.append((int(p[0]), int(p[1]), int(p[2])))
    return vertices, edges

def run_mst(_):
    with out_mst:
        clear_output(wait=True)
        try:
            vertices, edges = parse_graph()
            start = int(w_start.value.strip())
        except Exception as e:
            print(f'⚠  Input error: {e}'); return
        v = w_steps.value
        kruskal(vertices, edges, verbose=v)
        print()
        print('━' * 60)
        print()
        prim(vertices, edges, start=start, verbose=v)
        print()
        # quick agreement check
        from collections import defaultdict as dd
        import heapq as hq
        km, kc = kruskal(vertices, edges, verbose=False)
        pm, pc = prim(vertices, edges, start=start, verbose=False)
        match = '✓  Both algorithms agree.' if kc == pc else '✗  Cost mismatch — check input.'
        print(f'Cost check: Kruskal={kc}, Prim={pc}  |  {match}')

def gen_random(_):
    n = w_rand_n.value
    rv, re = random_graph(n)
    w_verts.value = ', '.join(map(str, rv))
    w_edges.value = '\n'.join(f'{u} {v} {w}' for u, v, w in re)
    w_start.value = str(rv[0])
    with out_mst:
        clear_output(wait=True)
        print(f'Random graph: {n} vertices, {len(re)} edges. Click Run.')

w_run.on_click(run_mst)
w_rand.on_click(gen_random)
display(widgets.VBox([
    widgets.HBox([w_rand_n, w_rand]),
    widgets.HBox([w_verts, w_start]),
    w_edges, w_steps, w_run, out_mst
]))


## Automated Test Cases

In [3]:
# ── AUTOMATED TEST CASES ──────────────────────────────────────────────────────
# Verifies that Kruskal's and Prim's always produce the same MST cost on
# the same graph. Also checks known hand-calculated examples.

import heapq as _hq
from collections import defaultdict as _dd

def _kruskal(v, e):
    vi={vx:i for i,vx in enumerate(v)}; uf=UnionFind(len(v)); mst=[]
    for u,vv,w in sorted(e,key=lambda x:x[2]):
        if uf.union(vi[u],vi[vv]): mst.append(w)
        if len(mst)==len(v)-1: break
    return sum(mst)
def _prim(v, e, start=None):
    adj=_dd(list)
    for u,vv,w in e: adj[u].append((vv,w)); adj[vv].append((u,w))
    s=v[0] if start is None else start
    vis=set(); heap=[(0,None,s)]; cost=0
    while heap:
        w,_,cur=_hq.heappop(heap)
        if cur in vis: continue
        vis.add(cur); cost+=w
        for nb,wt in adj[cur]:
            if nb not in vis: _hq.heappush(heap,(wt,cur,nb))
    return cost

MST_TESTS = [
    ('Spec example',
     [1,2,3,4,5], [(1,2,3),(2,4,5),(4,5,6),(1,3,4),(3,5,7)], 18),
    ('Simple triangle',
     [1,2,3], [(1,2,1),(2,3,2),(1,3,10)], 3),
    ('Linear chain',
     [1,2,3,4], [(1,2,5),(2,3,3),(3,4,4)], 12),
    ('All equal weights',
     [1,2,3,4], [(1,2,1),(2,3,1),(3,4,1),(1,4,1),(1,3,1),(2,4,1)], 3),
    ('6-vertex example (from spec)',
     [1,2,3,4,5,6],
     [(1,2,4),(1,3,3),(2,3,5),(2,4,6),(3,4,7),(3,5,8),(4,5,9),(4,6,5),(5,6,6)],
     24),
]

w_tst2 = widgets.Button(
    description='▶  Run MST Tests',
    button_style='info', layout=widgets.Layout(width='160px', height='36px')
)
out_tst2 = widgets.Output()

def run_mst_tests(_):
    with out_tst2:
        clear_output(wait=True)
        print('=== MST Automated Test Cases ===\n')
        total = passed = 0
        for name, verts, edges, expected in MST_TESTS:
            total += 1
            kc = _kruskal(verts, edges)
            pc = _prim(verts, edges)
            ok = (kc == pc == expected)
            if ok: passed += 1
            sym = '✓' if ok else '✗'
            print(f'  {sym}  {name}')
            print(f'       Expected={expected}  Kruskal={kc}  Prim={pc}')
            if not ok: print(f'       ← MISMATCH')
            print()
        print(f'Results: {passed}/{total} tests passed.')

w_tst2.on_click(run_mst_tests)
display(widgets.VBox([w_tst2, out_tst2]))
